# Chapter 9 Lab — Prompting and LLM Engineering

Runs a small open-weight instruction-tuned model locally through zero-shot, few-shot, and
chain-of-thought prompting, then tabulates a rough accuracy/latency comparison against a hosted
API call on the same task. Uses a small model so this runs on CPU; swap in a larger local model
or an API key for closer-to-production results.

In [ ]:
from transformers import pipeline
import time

# A small instruction-tuned model that runs locally on modest hardware.
generator = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")

## 1. Zero-shot vs. few-shot

In [ ]:
zero_shot = "Classify the sentiment as positive or negative: 'The food was cold and the service was slow.'"

few_shot = """Classify the sentiment as positive or negative.

Review: "Slow shipping but the product itself works great."
Sentiment: positive

Review: "Arrived broken and support never replied."
Sentiment: negative

Review: "The food was cold and the service was slow."
Sentiment:"""

for name, prompt in [("zero-shot", zero_shot), ("few-shot", few_shot)]:
    out = generator(prompt, max_new_tokens=5, do_sample=False)[0]["generated_text"]
    print(f"--- {name} ---\n{out}\n")

## 2. Chain-of-thought

In [ ]:
cot_prompt = ("Q: A store had 23 apples, sold 8, then received 15 more. How many apples now? "
              "Let's think step by step.")
print(generator(cot_prompt, max_new_tokens=60, do_sample=False)[0]["generated_text"])

## 3. Rough cost/latency comparison table

In [ ]:
import pandas as pd

start = time.time()
generator(few_shot, max_new_tokens=5, do_sample=False)
local_latency = time.time() - start

rows = [
    {"setup": "local small model", "latency_s": round(local_latency, 2), "per_call_cost": "$0 (own hardware)", "data_leaves_org": "No"},
    {"setup": "hosted API (illustrative)", "latency_s": "~0.5-2 (network+queue)", "per_call_cost": "per-token, provider-billed", "data_leaves_org": "Yes, unless enterprise/self-hosted option"},
]
pd.DataFrame(rows)

## Exercise

Fill in a real hosted-API call (any provider you have access to) in place of the illustrative
row above, and re-run the comparison on the same 10 sentiment examples used in Chapter 8's lab.
Which setup would you choose for a privacy-sensitive internal tool, and why?